# OD Model Benchmark

Comparison pipeline: **our GPS-based model** (pre-trained in `gps_od.ipynb`) vs
paper baselines and classical/graph/diffusion baselines.

**Our model:** GPS-GNN encoder + decoder, loaded from pre-trained weights.
**Benchmark baselines:** training and inference are separated in this notebook.
Train baseline artifacts once, then compare them against GPS via load-only inference.


In [ ]:
from pathlib import Path

import pandas as pd

from benchmarking.config import (
    BASELINE_MODELS,
    DATA_PATH,
    GPS_BENCHMARK_IDS,
    GPS_MC_BENCHMARK_IDS,
    MULTI_CITY_IDS,
    PROJECT_ROOT,
    SINGLE_CITY_ID,
    TRANSFLOWER_ORIG_CONFIG,
    WEIGHTS_DIR,
    cleanup_gpu,
    device,
    set_global_seed,
    trained_gps_run_ids,
    trained_lgbm_run_ids,
    GMEL_GPS_BENCHMARK_IDS,
    trained_gmel_gps_run_ids,
)
from benchmarking.data_utils import get_all_areas
from benchmarking.gps_loader import GPSBenchmarkLoader
from benchmarking.pipeline import run_multi_city_benchmark, run_single_city_benchmark
from benchmarking.reporting import (
    build_combined_summary,
    plot_comparison,
    results_to_dataframe,
    save_results_table,
)

set_global_seed()

trained_sc = trained_gps_run_ids(GPS_BENCHMARK_IDS)
trained_mc = trained_gps_run_ids(GPS_MC_BENCHMARK_IDS)
LGBM_BENCHMARK_IDS = trained_lgbm_run_ids(GPS_BENCHMARK_IDS)
gmel_gps_run_ids = trained_gmel_gps_run_ids()
gps_loader = GPSBenchmarkLoader(single_city_id=SINGLE_CITY_ID, multi_city_ids=MULTI_CITY_IDS, data_path=DATA_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")
print(f"Data path: {DATA_PATH}")


Project root: /home/rk/Документы/Python Projects/GSP_OD_Prediction
Device: cuda:1
Data path: /home/rk/Документы/Python Projects/GSP_OD_Prediction/data


## Configurable Model List
Comment/uncomment to select which models to run.

In [ ]:
print(f"GPS (SC) run_ids: {len(GPS_BENCHMARK_IDS)}  ({len(trained_sc)} fully trained across {len(SINGLE_CITY_IDS)} cities)")
print(f"GPS (MC) run_ids: {len(GPS_MC_BENCHMARK_IDS)}  ({len(trained_mc)} trained)")
print(f"LGBM donor ids:  {len(LGBM_BENCHMARK_IDS)}")
print(f"GMEL_GPS ids:    {len(gmel_gps_run_ids)}")
print(f"SC cities:       {SINGLE_CITY_IDS}")
print(f"Inference seeds: {INFERENCE_SEEDS}")
print(f"Baseline models: {len(BASELINE_MODELS)}")
print(f"SC baseline artifacts ready: {len(trained_sc_baselines)} / {len(BASELINE_MODELS)}")
print(f"MC baseline artifacts ready: {len(trained_mc_baselines)} / {len(BASELINE_MODELS)}")
print(f"SC baseline ready ids: {trained_sc_baselines}")
print(f"MC baseline ready ids: {trained_mc_baselines}")
print(f"TransFlower orig config: {TRANSFLOWER_ORIG_CONFIG.describe()}")


GPS (SC) run_ids: 12  (0 trained)
GPS (MC) run_ids: 7  (0 trained)
LGBM run_ids:     0
Baseline models:  2
TransFlower orig config: MLP+transflower+ce+normalized | pe=rwpe | norm=batch_norm | RLE | zeros=True samp=False


## Single-City Benchmark

All models predict on city `48201`.
Training and inference are separated:
1. train/save baseline artifacts once
2. run inference-only comparison against pre-trained GPS variants


In [ ]:
# Single-city baseline training helper, split by benchmark city like in gps_od.ipynb
SC_CITY_1, SC_CITY_2, SC_CITY_3 = SINGLE_CITY_IDS
print(f"Single-city benchmark cities: {SINGLE_CITY_IDS}")

def train_sc_baseline_city(area_id):
    print(f"\n{'=' * 70}\n  SINGLE-CITY BASELINE TRAINING FOR {area_id}\n{'=' * 70}")
    city_runs = train_single_city_benchmark_models(
        single_city_ids=[area_id],
        data_path=DATA_PATH,
        baseline_models=BASELINE_MODELS,
        gps_loader=gps_loader,
    )
    single_city_baseline_train_runs[area_id] = city_runs
    trained_city_baselines = trained_single_city_baseline_models(BASELINE_MODELS, [area_id])
    print(f"\nArtifacts ready for {area_id}: {trained_city_baselines}")
    print(f"Training calls completed for {area_id}: {list(city_runs)}")
    return city_runs


In [ ]:
sc_city_1_baseline_runs = train_sc_baseline_city(SC_CITY_1)


Benchmark helpers loaded from benchmarking/*.py
Reusable model entrypoints: DGM, GM_E, GM_P, GMEL


In [ ]:
sc_city_2_baseline_runs = train_sc_baseline_city(SC_CITY_2)


Graph-family runner is provided by benchmarking.runners.run_graph_model


In [ ]:
sc_city_3_baseline_runs = train_sc_baseline_city(SC_CITY_3)


Diffusion-family runner is provided by benchmarking.runners.run_diffusion_model


In [ ]:
trained_sc_baselines = trained_single_city_baseline_models(BASELINE_MODELS, SINGLE_CITY_IDS)
print(f"\nArtifacts ready across {SINGLE_CITY_IDS}: {trained_sc_baselines}")
print(f"Per-city training calls completed for: {list(single_city_baseline_train_runs)}")


GPS and GPS+LGBM loading is provided by benchmarking.gps_loader.GPSBenchmarkLoader


In [ ]:
print("=" * 70)
print("  SINGLE-CITY BENCHMARK INFERENCE")
print(f"  Cities: {SINGLE_CITY_IDS}")
print(f"  Inference seeds: {INFERENCE_SEEDS}")
print("=" * 70)

trained_sc_baselines = trained_single_city_baseline_models(BASELINE_MODELS, SINGLE_CITY_IDS)
print(f"Baseline artifacts available: {trained_sc_baselines}")

single_city_results, single_city_model_type = run_single_city_benchmark(
    gps_run_ids=GPS_BENCHMARK_IDS,
    lgbm_run_ids=LGBM_BENCHMARK_IDS,
    single_city_ids=SINGLE_CITY_IDS,
    data_path=DATA_PATH,
    baseline_models=BASELINE_MODELS,
    gps_loader=gps_loader,
    gmel_gps_run_ids=gmel_gps_run_ids,
    inference_seeds=INFERENCE_SEEDS,
)

All models predict on city `48201`. OD pairs split 80/10/10 for training.

In [8]:
print("=" * 70)
print("  SINGLE-CITY BENCHMARK")
print(f"  City: {SINGLE_CITY_ID}")
print("=" * 70)

single_city_results, single_city_model_type = run_single_city_benchmark(
    gps_run_ids=GPS_BENCHMARK_IDS,
    lgbm_run_ids=LGBM_BENCHMARK_IDS,
    single_city_id=SINGLE_CITY_ID,
    data_path=DATA_PATH,
    baseline_models=BASELINE_MODELS,
    gps_loader=gps_loader,
    gmel_gps_run_ids=gmel_gps_run_ids,
)

print(f"\nCompleted: {len(single_city_results)} models")


  SINGLE-CITY BENCHMARK
  City: 48201

[Our Model - GPS variants]
  [SKIP] SC_TF_CE: weights not found at /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/weights/SC_TF_CE.pt
  [SKIP] SC_TF_H: weights not found at /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/weights/SC_TF_H.pt
  [SKIP] SC_TF_CE_lape: weights not found at /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/weights/SC_TF_CE_lape.pt
  [SKIP] SC_TF_CE_gn: weights not found at /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/weights/SC_TF_CE_gn.pt
  [SKIP] SC_TF_focal: weights not found at /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/weights/SC_TF_focal.pt
  [SKIP] SC_TF_CE_samp: weights not found at /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/weights/SC_TF_CE_samp.pt
  [SKIP] SC_TF_CE_nz: weights not found at /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/weights/SC_TF_CE_nz.pt
  [SKIP] SC_TF_CE_rle: weights not found at /home/r

GMEL-GAT: 100%|██████████| 10/10 [00:01<00:00,  9.44ep/s, loss=25.66, pat=99, val=26.16]


  -> Loss plot saved to /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/loss_plots/GMEL_Encoder_Loss_20260404_220559_loss.png

  === Bilinear Head ===
    CPC_full=0.0878  CPC_nz=0.1835  CPC_test=0.1818  MAE=39.1957  RMSE=51.6683
    Full metrics: {'num_regions': 786, 'RMSE': 51.6683349609375, 'NRMSE': 4.906224250793457, 'MAE': 39.19570541381836, 'MAPE': 27.95639991760254, 'SMAPE': 1.6810941696166992, 'CPC': 0.08779152482748032, 'RMSE_nonzero': 48.90079879760742, 'MAE_nonzero': 36.135616302490234, 'MAPE_nonzero': 11.935425758361816, 'SMAPE_nonzero': 1.5675044059753418, 'CPC_nonzero': 0.18352110683918, 'accuracy': 0.47353333462825914, 'matrix_COS_similarity': 0.2872214913368225, 'JSD_inflow': 0.6507856707645657, 'JSD_outflow': 0.992963105662133, 'JSD_ODflow': 0.49852100985649184}
    Nonzero metrics: {'CPC': 0.18352110683918, 'MAE': 36.135616302490234, 'RMSE': 48.90079879760742}
    Test-pair metrics: {'CPC': 0.18178409337997437, 'MAE': 35.92778015136719, 'RMSE': 48.5489501

GMEL-GAT: 100%|██████████| 10/10 [00:00<00:00, 20.51ep/s, loss=25.63, pat=99, val=26.15]


  -> Loss plot saved to /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/loss_plots/GMEL_Encoder_Loss_20260404_220723_loss.png

  === Bilinear Head ===
    CPC_full=0.0000  CPC_nz=0.0000  CPC_test=0.0000  MAE=2.5659  RMSE=10.8393
    Full metrics: {'num_regions': 786, 'RMSE': 10.839269638061523, 'NRMSE': 1.0292549133300781, 'MAE': 2.565918207168579, 'MAPE': 0.3187375068664551, 'SMAPE': 0.9288600087165833, 'CPC': 0.0, 'RMSE_nonzero': 15.90522575378418, 'MAE_nonzero': 5.524876117706299, 'MAPE_nonzero': 0.6862980127334595, 'SMAPE_nonzero': 2.0, 'CPC_nonzero': 0.0, 'accuracy': 0.5355699939785948, 'matrix_COS_similarity': 0.0, 'JSD_inflow': 1.0, 'JSD_outflow': 0.9999999999999998, 'JSD_ODflow': 0.28367948818790956}
    Nonzero metrics: {'CPC': 0.0, 'MAE': 5.524876117706299, 'RMSE': 15.90522575378418}
    Test-pair metrics: {'CPC': 0.0, 'MAE': 5.327606201171875, 'RMSE': 14.425789833068848}
  GMEL: fitting LGBM on embeddings...
Training until validation scores don't improve for 50 

In [9]:
if single_city_results:
    df_single = results_to_dataframe(single_city_results, single_city_model_type)

    print("\n" + "=" * 100)
    print("  SINGLE-CITY RESULTS (3 cities, mean/std, sorted by CPC)")
    print("=" * 100)
    print(df_single.to_string())

    single_city_path = save_results_table(df_single, "single_city_benchmark.csv")
    print(f"\nSaved to {single_city_path}")
else:
    print("No results to display.")



  SINGLE-CITY RESULTS (sorted by CPC)
                CPC   CPC_val  CPC_full  CPC_nonzero    CPC_nz  CPC_test      RMSE       MAE      MAPE     SMAPE  MAE_test  RMSE_test  accuracy  matrix_COS_similarity  JSD_inflow  JSD_outflow  JSD_ODflow model_type
GMEL       0.710339  0.772194  0.710338     0.804316  0.804316  0.771482  3.591812  1.685710  0.884043  1.182180  2.457688   6.409203  0.666866               0.838851    0.207232     0.059447    0.039802   Baseline
GMEL_LGBM  0.709422  0.771550  0.709423     0.802175  0.802175  0.771595  3.649271  1.689668  0.878070  1.176526  2.460379   6.453096  0.670200               0.839105    0.200068     0.063014    0.037323   Baseline

Saved to /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/single_city_benchmark.csv


## Multi-City (10 Cities) Benchmark

10 cities, area-level split: 8 train, 1 val, 1 test.

In [10]:
BASELINE_MODELS = [ 'DGM', 'GMEL', 'GMEL_LGBM']

In [ ]:
print("=" * 70)
print("  MULTI-CITY BENCHMARK INFERENCE")
print(f"  Cities: {MULTI_CITY_IDS}")
print(f"  Inference seeds: {INFERENCE_SEEDS}")
print("=" * 70)

trained_mc_baselines = trained_multi_city_baseline_models(BASELINE_MODELS)
print(f"Baseline artifacts available: {trained_mc_baselines}")

multi_city_results, multi_city_model_type, multi_city_split = run_multi_city_benchmark(
    gps_run_ids=GPS_MC_BENCHMARK_IDS,
    city_ids=MULTI_CITY_IDS,
    data_path=DATA_PATH,
    baseline_models=BASELINE_MODELS,
    gps_loader=gps_loader,
    inference_seeds=INFERENCE_SEEDS,
)

print(f"Train: {multi_city_split['train']}")
print(f"Valid: {multi_city_split['valid']}")
print(f"Test:  {multi_city_split['test']}")
print(f"\nCompleted: {len(multi_city_results)} models")


  MULTI-CITY BENCHMARK
  Cities: ['17031', '48201', '04013', '06073', '06059', '36047', '12086', '48113', '06065', '36081']

[Our Model - GPS variants]
  [SKIP] MC_TF_CE: config JSON not found.
  [SKIP] MC_TF_H: config JSON not found.
  [SKIP] MC_TF_CE_lape: config JSON not found.
  [SKIP] MC_TF_focal: config JSON not found.
  [SKIP] MC_TF_CE_rle: config JSON not found.
  [SKIP] MC_BL_CE: config JSON not found.
  [SKIP] MC_BL_H: config JSON not found.

[Baseline - TransFlower orig (paper)]
  17031: N=1318
  48201: N=786
  04013: N=916
  06073: N=627
  06059: N=582
  36047: N=761
  12086: N=518
  48113: N=529
  06065: N=453
  36081: N=669
  06065: N=453
  48201: N=786
  36047: N=761
  17031: N=1318
  48113: N=529
  04013: N=916
  36081: N=669
  06059: N=582
  06073: N=627
  12086: N=518

  MC TransFlower Orig (MLP+TF+RLE)
  MLP+transflower+ce+normalized | pe=rwpe | norm=batch_norm | RLE | zeros=True samp=False
  Params: 517,699
  [diag] encoder=mlp decoder=transflower pe=rwpe norm=batch

In [ ]:
if multi_city_results:
    df_multi = results_to_dataframe(multi_city_results, multi_city_model_type)

    print("\n" + "=" * 100)
    print("  MULTI-CITY RESULTS (mean/std, sorted by CPC)")
    print("=" * 100)
    print(df_multi.to_string())

    multi_city_path = save_results_table(df_multi, "multi_city_benchmark.csv")
    print(f"\nSaved to {multi_city_path}")
else:
    print("No results to display.")


## Visualization

In [ ]:
plot_comparison(single_city_results, f"Single-City Benchmark ({', '.join(SINGLE_CITY_IDS)})", ["CPC", "MAE", "RMSE"])
plot_comparison(multi_city_results, "Multi-City Benchmark (10 cities)", ["CPC", "MAE", "RMSE"])


In [ ]:
print("\n" + "=" * 120)
print("  COMBINED SUMMARY")
print("=" * 120)

df_summary = build_combined_summary(single_city_results, multi_city_results)
print(df_summary.to_string())

summary_path = save_results_table(df_summary, "benchmark_combined.csv")
print(f"\nSaved to {summary_path}")
